In [88]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity


In [50]:
path_to_data = Path('movie_data/')
for file_path in path_to_data.glob('**/*'):
    if file_path.is_file():
        print(file_path)

movie_data\genome_scores.csv
movie_data\genome_tags.csv
movie_data\link.csv
movie_data\movie.csv
movie_data\rating.csv
movie_data\tag.csv


In [51]:
movie = pd.read_csv('movie_data/movie.csv')
rating = pd.read_csv('movie_data/rating.csv')
link = pd.read_csv('movie_data/link.csv')
tag = pd.read_csv('movie_data/tag.csv')
genome_tags = pd.read_csv('movie_data/genome_tags.csv')
genome_scores = pd.read_csv('movie_data/genome_scores.csv')

In [77]:
x = rating['userId'].value_counts() > 500
print(len(x[x]))
thin_rating = rating[rating['userId'].isin(x[x].index)]
y = thin_rating['movieId'].value_counts() > 50
print(len(y[y]))
thin_rating = thin_rating[thin_rating['movieId'].isin(y[y].index)]
movie_pivot = thin_rating.pivot_table(columns='userId', index='movieId', values='rating')

7441
9313


In [78]:
movie_pivot.fillna(0, inplace=True)
movie_pivot

userId,11,24,54,58,91,104,116,134,156,208,...,138270,138301,138307,138325,138382,138397,138406,138411,138437,138474
movieId,,,,,,,,,,,,,,,,,,,,,
1,4.5,4.0,4.0,5.0,4.0,0.0,3.0,4.0,5.0,4.0,...,0.0,2.5,3.5,5.0,3.0,0.0,4.0,5.0,4.0,5.0
2,0.0,0.0,3.0,0.0,3.5,0.0,2.0,0.0,5.0,0.0,...,0.0,2.5,2.5,3.0,4.0,0.0,3.0,0.0,0.0,4.0
3,0.0,0.0,0.0,0.0,3.0,0.0,2.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,3.0,5.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,...,1.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0
5,0.0,2.0,3.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118700,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
118900,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
118997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
movie_sparse = csr_matrix(movie_pivot)
model = NearestNeighbors(n_neighbors=10, algorithm='brute', metric='cosine')
model.fit(movie_sparse)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=10)

In [80]:
distances, suggestions = model.kneighbors(movie_pivot.iloc[1,:].values.reshape(1,-1))
distances, suggestions

(array([[0.        , 0.22626448, 0.23797185, 0.23983944, 0.24134662,
         0.24469272, 0.2489989 , 0.25214387, 0.2534928 , 0.25541199]]),
 array([[   1,  438,  658, 1273,  326,  334,  337, 2112, 1037,  532]]))

In [81]:
similarities = 1 - distances[0][1:]
neighbor_indices = suggestions[0][1:]

numerator = 0
denominator = 0

for k, idx in enumerate(neighbor_indices):
    mid = movie_pivot.index[idx]
    r_jx = movie_pivot.loc[mid, 24]  # User 24의 실제 평점
    if r_jx == 0:
        continue
    numerator += similarities[k] * r_jx   # s_ij × r_jx
    denominator += similarities[k]          # s_ij

pred = numerator / denominator
print(f"1-1) Cosine weighted avg 예측: {pred:.4f}")

1-1) Cosine weighted avg 예측: 4.2817


In [ ]:
target_user = 24
target_item = 2

item_idx = movie_pivot.index.get_loc(target_item)
item_vec = movie_pivot.iloc[item_idx].values
pearson_sims = []

for i in range(len(movie_pivot)):
    if i == item_idx:
        pearson_sims.append((i, -999))
        continue
    other_vec = movie_pivot.iloc[i].values
    both_rated = (item_vec != 0) & (other_vec != 0)
    if both_rated.sum() >= 2:
        corr, _ = pearsonr(item_vec[both_rated], other_vec[both_rated])
        pearson_sims.append((i, corr))
    else:
        pearson_sims.append((i, -999))

pearson_sims.sort(key=lambda x: x[1], reverse=True)
topK = pearson_sims[:10]
neighbor_indices = [x[0] for x in topK]
similarities = np.array([x[1] for x in topK])

numerator = 0
denominator = 0

for k, idx in enumerate(neighbor_indices):
    mid = movie_pivot.index[idx]
    r_xj = movie_pivot.loc[mid, target_user]
    
    if r_xj == 0:
        continue
    
    sim = similarities[k]
    
    numerator += sim * r_xj
    denominator += abs(sim)  
if denominator == 0:
    pred = movie_pivot.values[movie_pivot.values > 0].mean()
else:
    pred = numerator / denominator
print(f"1-2) Pearson weighted avg 예측: {pred:.4f}")

1-2) Pearson weighted avg 예측: 3.3468


In [ ]:
target_user = 24
target_item = 2


user_vec = movie_pivot.loc[:, target_user].values
item_raters = movie_pivot.columns[movie_pivot.loc[target_item] != 0]

pearson_user_sims = []
for uid in item_raters:
    if uid == target_user:
        continue
    
    other_vec = movie_pivot.loc[:, uid].values
    
    both_rated = (user_vec != 0) & (other_vec != 0)
    
    if both_rated.sum() >= 2:
        corr, _ = pearsonr(user_vec[both_rated], other_vec[both_rated])
        pearson_user_sims.append((uid, corr))

pearson_user_sims.sort(key=lambda x: x[1], reverse=True)
topK = pearson_user_sims[:10]
neighbor_uids = [x[0] for x in topK]
similarities = np.array([x[1] for x in topK])

numerator = 0
denominator = 0

for k, uid in enumerate(neighbor_uids):
    r_yj = movie_pivot.loc[target_item, uid]  
    if r_yj == 0:
        continue
    sim = similarities[k]
    numerator += sim * r_yj
    denominator += abs(sim) 

if denominator == 0:
    pred = movie_pivot.values[movie_pivot.values > 0].mean()
else:
    pred = numerator / denominator

print(f"1-3) User-User Pearson weighted avg 예측: {pred:.4f}")

1-3) User-User Pearson weighted avg 예측: 2.8184


In [68]:
np.random.seed(42)

rated_pairs = []
for mid in movie_pivot.index:
    for uid in movie_pivot.columns:
        val = movie_pivot.loc[mid, uid]
        if val != 0:
            rated_pairs.append((uid, mid, val))
rated_df = pd.DataFrame(rated_pairs, columns=['userId', 'movieId', 'rating'])
test_samples = rated_df.sample(n=1000, random_state=42).reset_index(drop=True)
print(test_samples.head())
print(f"총 {len(test_samples)}개 샘플")

   userId  movieId  rating
0   17918    26151     5.0
1   27267    64614     5.0
2   50575     2565     4.0
3   80542     1320     3.0
4   59568      543     4.0
총 1000개 샘플


In [ ]:
movie_pivot_train = movie_pivot.copy()
for _, row in test_samples.iterrows():
    movie_pivot_train.loc[row['movieId'], row['userId']] = 0

movie_pivot_train_nan = movie_pivot_train.replace(0, np.nan)
mr = movie_pivot_train_nan.stack().mean()

user_index = {uid: idx for idx, uid in enumerate(movie_pivot_train.columns)}
movie_index = {mid: idx for idx, mid in enumerate(movie_pivot_train.index)}

print("Cosine similarity 계산 중...")
user_cos_sim = cosine_similarity(movie_pivot_train.T.values)
item_cos_sim = cosine_similarity(movie_pivot_train.values)
print("완료!")

print("Pearson similarity 계산 중...")
user_pearson_sim = np.corrcoef(movie_pivot_train.T.fillna(0))
item_pearson_sim = np.corrcoef(movie_pivot_train.fillna(0))

user_pearson_sim = np.nan_to_num(user_pearson_sim)
item_pearson_sim = np.nan_to_num(item_pearson_sim)
print("완료!")

K = 10
def predict_user_cosine_simple(uid, mid):
    uid_idx = user_index[uid]
    mid_idx = movie_index[mid]

    sim_scores = user_cos_sim[uid_idx].copy()
    sim_scores[uid_idx] = -1
    neighbor_idx = np.argsort(sim_scores)[-K:]

    ratings = [movie_pivot_train.iloc[mid_idx, idx] 
            for idx in neighbor_idx if movie_pivot_train.iloc[mid_idx, idx] != 0]

    return np.mean(ratings) if ratings else mr


def predict_user_cosine_weighted(uid, mid):
    uid_idx = user_index[uid]
    mid_idx = movie_index[mid]

    sim_scores = user_cos_sim[uid_idx].copy()
    sim_scores[uid_idx] = -1
    neighbor_idx = np.argsort(sim_scores)[-K:]

    num, den = 0, 0
    for idx in neighbor_idx:
        r = movie_pivot_train.iloc[mid_idx, idx]
        if r == 0:
            continue
        s = sim_scores[idx]
        num += s * r
        den += abs(s)

    return num / den if den != 0 else mr


def predict_user_pearson_simple(uid, mid):
    uid_idx = user_index[uid]
    mid_idx = movie_index[mid]

    sim_scores = user_pearson_sim[uid_idx].copy()
    sim_scores[uid_idx] = -1
    neighbor_idx = np.argsort(sim_scores)[-K:]

    ratings = [movie_pivot_train.iloc[mid_idx, idx] 
            for idx in neighbor_idx if movie_pivot_train.iloc[mid_idx, idx] != 0]

    return np.mean(ratings) if ratings else mr


def predict_user_pearson_weighted(uid, mid):
    uid_idx = user_index[uid]
    mid_idx = movie_index[mid]

    sim_scores = user_pearson_sim[uid_idx].copy()
    sim_scores[uid_idx] = -1
    neighbor_idx = np.argsort(sim_scores)[-K:]

    num, den = 0, 0
    for idx in neighbor_idx:
        r = movie_pivot_train.iloc[mid_idx, idx]
        if r == 0:
            continue
        s = sim_scores[idx]
        num += s * r
        den += abs(s)

    return num / den if den != 0 else mr


def predict_item_cosine_simple(uid, mid):
    uid_idx = user_index[uid]
    mid_idx = movie_index[mid]

    sim_scores = item_cos_sim[mid_idx].copy()
    sim_scores[mid_idx] = -1
    neighbor_idx = np.argsort(sim_scores)[-K:]

    ratings = [movie_pivot_train.iloc[idx, uid_idx] 
            for idx in neighbor_idx if movie_pivot_train.iloc[idx, uid_idx] != 0]

    return np.mean(ratings) if ratings else mr


def predict_item_cosine_weighted(uid, mid):
    uid_idx = user_index[uid]
    mid_idx = movie_index[mid]

    sim_scores = item_cos_sim[mid_idx].copy()
    sim_scores[mid_idx] = -1
    neighbor_idx = np.argsort(sim_scores)[-K:]

    num, den = 0, 0
    for idx in neighbor_idx:
        r = movie_pivot_train.iloc[idx, uid_idx]
        if r == 0:
            continue
        s = sim_scores[idx]
        num += s * r
        den += abs(s)

    return num / den if den != 0 else mr


def predict_item_pearson_simple(uid, mid):
    uid_idx = user_index[uid]
    mid_idx = movie_index[mid]

    sim_scores = item_pearson_sim[mid_idx].copy()
    sim_scores[mid_idx] = -1
    neighbor_idx = np.argsort(sim_scores)[-K:]

    ratings = [movie_pivot_train.iloc[idx, uid_idx] 
            for idx in neighbor_idx if movie_pivot_train.iloc[idx, uid_idx] != 0]

    return np.mean(ratings) if ratings else mr


def predict_item_pearson_weighted(uid, mid):
    uid_idx = user_index[uid]
    mid_idx = movie_index[mid]

    sim_scores = item_pearson_sim[mid_idx].copy()
    sim_scores[mid_idx] = -1
    neighbor_idx = np.argsort(sim_scores)[-K:]

    num, den = 0, 0
    for idx in neighbor_idx:
        r = movie_pivot_train.iloc[idx, uid_idx]
        if r == 0:
            continue
        s = sim_scores[idx]
        num += s * r
        den += abs(s)

    return num / den if den != 0 else mr


methods = [
    ("User", "Cosine",  "Simple",   predict_user_cosine_simple),
    ("User", "Cosine",  "Weighted", predict_user_cosine_weighted),
    ("User", "Pearson", "Simple",   predict_user_pearson_simple),
    ("User", "Pearson", "Weighted", predict_user_pearson_weighted),
    ("Item", "Cosine",  "Simple",   predict_item_cosine_simple),
    ("Item", "Cosine",  "Weighted", predict_item_cosine_weighted),
    ("Item", "Pearson", "Simple",   predict_item_pearson_simple),
    ("Item", "Pearson", "Weighted", predict_item_pearson_weighted),
]

results = []
for cf, metric, avg, fn in methods:
    print(f"\n실행: {cf} / {metric} / {avg}")
    errors = []

    for idx, row in test_samples.iterrows():
        pred = fn(row['userId'], row['movieId'])
        errors.append((pred - row['rating']) ** 2)

        if (idx+1) % 200 == 0:
            print(f"  {idx+1}/1000")

    rmse = np.sqrt(np.mean(errors))
    results.append({'CF': cf, 'Metric': metric, 'Avg': avg, 'RMSE': round(rmse, 4)})
    print(f"  RMSE: {rmse:.4f}")

print("\n" + "="*50)
print(pd.DataFrame(results).to_string(index=False))

Cosine similarity 계산 중...
완료!
Pearson similarity 계산 중...
완료!

실행: User / Cosine / Simple
  200/1000
  400/1000
  600/1000
  800/1000
  1000/1000
  RMSE: 0.9337

실행: User / Cosine / Weighted
  200/1000
  400/1000
  600/1000
  800/1000
  1000/1000
  RMSE: 0.9321

실행: User / Pearson / Simple
  200/1000
  400/1000
  600/1000
  800/1000
  1000/1000
  RMSE: 0.9367

실행: User / Pearson / Weighted
  200/1000
  400/1000
  600/1000
  800/1000
  1000/1000
  RMSE: 0.9347

실행: Item / Cosine / Simple
  200/1000
  400/1000
  600/1000
  800/1000
  1000/1000
  RMSE: 0.8900

실행: Item / Cosine / Weighted
  200/1000
  400/1000
  600/1000
  800/1000
  1000/1000
  RMSE: 0.8878

실행: Item / Pearson / Simple
  200/1000
  400/1000
  600/1000
  800/1000
  1000/1000
  RMSE: 0.8289

실행: Item / Pearson / Weighted
  200/1000
  400/1000
  600/1000
  800/1000
  1000/1000
  RMSE: 0.8266

  CF  Metric      Avg   RMSE
User  Cosine   Simple 0.9337
User  Cosine Weighted 0.9321
User Pearson   Simple 0.9367
User Pearson Weigh

In [ ]:
errors = []
for i, row in test_samples.iterrows():
    uid = row['userId']
    mid = row['movieId']
    actual = row['rating']
    
    mid_idx = movie_pivot_train.index.get_loc(mid)
    target_vec = movie_pivot_train.iloc[mid_idx].values
    
    user_rated_mids = movie_pivot_train.index[movie_pivot_train.loc[:, uid] != 0]
    
    pearson_list = []
    for n_mid in user_rated_mids:
        if n_mid == mid:
            continue
        n_idx = movie_pivot_train.index.get_loc(n_mid)
        other_vec = movie_pivot_train.iloc[n_idx].values
        both = (target_vec != 0) & (other_vec != 0)
        if both.sum() >= 2:
            corr, _ = pearsonr(target_vec[both], other_vec[both])
            pearson_list.append((n_mid, corr))
    
    if len(pearson_list) == 0:
        continue
    
    pearson_list.sort(key=lambda x: x[1], reverse=True)
    topK = pearson_list[:K]
    
    b_x = movie_pivot_train_nan.loc[:, uid].mean() - mr
    b_i = movie_pivot_train_nan.loc[mid].mean() - mr
    bl = mr + b_x + b_i
    
    numerator = 0
    denominator = 0
    
    for n_mid, corr in topK:
        if corr <= 0:
            continue
        r = movie_pivot_train.loc[n_mid, uid]
        b_j = movie_pivot_train_nan.loc[n_mid].mean() - mr
        b_xj = mr + b_x + b_j
        numerator += corr * (r - b_xj)
        denominator += corr
    
    if denominator == 0:
        continue
    
    pred = bl + numerator / denominator
    errors.append((pred - actual) ** 2)

rmse = np.sqrt(np.mean(errors))
print(f"Item-based + Pearson + Weighted avg RMSE: {rmse:.4f}")
print(f"예측 성공: {len(errors)}/1000")